In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# How to Compute the Conjunctive Normal Form

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [ ]:
import { LogicParser, Formula } from './PropositionalLogicParser'

In [ ]:
type Variable = string;
type Formula = Variable | Formula[];
type Literal = Variable | [string, Variable];
type Clause = Set<Literal>;
type CNF = Set<Clause>;

In [ ]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

The function `eliminateBiconditional(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$

In [ ]:
function eliminateBiconditional(f: Formula): Formula | null {
    //Eliminate the logical operator "→" from f.
    if (typeof f === 'string') { // This case covers variables.
        return f;
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '↔': {
                const [g, h] = args as [Formula, Formula];
                return eliminateBiconditional(['∧', ['→', g, h], ['→', h, g]]);
            }
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateBiconditional(g)];
            }
            case '→':
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
            }
        }
    }
    return null;
}

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from propositional logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$

In [ ]:
function eliminateConditional(f: Formula): Formula | null {
    //Eliminate the logical operator "→" from f.
    if (typeof f === 'string') { // variables
        return f; 
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
            case '⊥':
                return f;
            case '→': {
                const [g, h] = args as [Formula, Formula];
                return eliminateConditional(['∨', ['¬', g], h]);
            }
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateConditional(g)];
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateConditional(g), eliminateConditional(h)];
            }
        }
    }
    return null;
}

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

The forward declaration for the function `neg` is needed to typecheck the function `nnf`.

In [ ]:
let neg = (f: Formula): Formula | null=> {
    return null;
};

In [ ]:
function nnf(f: Formula): Formula | null {
    // Compute the negation normal form of f.
    if (typeof f === 'string') {
        return f;
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return neg(g);
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, nnf(g), nnf(h)];
            }
        }
    }
    return null;
}

In [ ]:
neg = function(f: Formula): Formula | null {
    // Compute the negation normal form of ¬f.
    if (typeof f === 'string') {
        return ['¬', f];
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
                return ['⊥'];
            case '⊥':
                return ['⊤'];
            case '¬': {
                const [g] = args as [Formula];
                return nnf(g);
            }
            case '∧': {
                const [g, h] = args as [Formula, Formula];
                return ['∨', neg(g), neg(h)];
            }
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return ['∧', neg(g), neg(h)];
            }
        }
    }
    return null;
}

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

In [ ]:
function cnf(f: Formula): CNF | null {
    if (typeof f === 'string') { // f is a variable
        return new Set([new Set([f])]);
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
                return new Set();
            case '⊥':
                return new Set([new Set()]);
            case '¬': {
                const [p] = args as [Variable];
                return new Set([new Set([['¬', p]])]); // f is a negative literal
            }
            case '∧': {
                const [g, h] = args as [Formula, Formula];
                const left = cnf(g);
                const right = cnf(h);
                return new Set([...left, ...right]);
            }
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                const left = cnf(g);
                const right = cnf(h);
                const result = new Set<Clause>();
                for (const k1 of left) {
                    for (const k2 of right) {
                        const unionClause = new Set([...k1, ...k2]);
                        result.add(unionClause);
                    }
                }
                return result;
            }
        }
    }
    return null;
}

The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [ ]:
function isTrivial(clause: Clause): boolean {
    for (const p of clause) {
        if (typeof p === 'string') {
            if (clauseHasLiteral(clause, ['¬', p])) {
                return true;
            }
        } else if (Array.isArray(p) && p.length === 2 && p[0] === '¬') {
            if (clause.has(p[1])) {
                return true;
            }
        }
    }
    return false;
}
function clauseHasLiteral(clause: Clause, literal: Literal): boolean {
    for (const lit of clause) {
        if (isLiteralEqual(lit, literal)) {
            return true;
        }
    }
    return false;
}
function isLiteralEqual(a: Literal, b: Literal): boolean {
    if (typeof a === 'string' && typeof b === 'string') {
        return a === b;
    }
    if (Array.isArray(a) && Array.isArray(b)) {
        return a[0] === b[0] && a[1] === b[1];
    }
    return false;
}

The function `removeDuplicatesFromClause` is necessary because in TypeScript, sets compare objects (like tuples representing literals) by reference, not by value. This means duplicates of structurally identical literals can exist as different objects in a set. The function converts each literal to a canonical string form, uses this string as a key to detect duplicates, and rebuilds the clause with only unique literals. This ensures clauses in CNF do not contain redundant duplicates.

In [ ]:
function removeDuplicatesFromClause(clause: Set<Literal>): Clause {
    const literalsSeen = new Set<string>();
    const result = new Set<Literal>();
    for (const lit of clause) {
        const key = literalToString(lit);
        if (!literalsSeen.has(key)) {
            literalsSeen.add(key);
            result.add(lit);
        }
    }
    return result;
}

function literalToString(lit: Literal): string {
    if (typeof lit === 'string') {
        return lit;
    } else {
        return `(${lit[0]},${lit[1]})`;
    }
}

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [ ]:
function removeDuplicatesFromSetsOfClauses(clauses: Set<Clause>): Set<Clause> {
    const clauseSeen = new Set<string>();
    const result = new Set<Clause>();
    for (const clause of clauses) {
        const cleanedClause = removeDuplicatesFromClause(clause);
        const key = clauseToString(cleanedClause);

        if (!clauseSeen.has(key)) {
            clauseSeen.add(key);
            result.add(cleanedClause);
        }
    }
    return result;
}

function clauseToString(clause: Clause): string {
    const sorted = [...clause]
        .map(literalToString)
        .sort()
        .join(',');
    return `{${sorted}}`;
}

In [ ]:
function simplify(clauses: Set<Clause>): Set<Clause> {
    const result = new Set<Clause>();
    for (const C of clauses) {
        if (!isTrivial(C)) {
            result.add(C);
        }
    }
    return removeDuplicatesFromSetsOfClauses(result);
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, trivial clausues are removed.

In [ ]:
function normalize(f: Formula): CNF {
    const n1 = eliminateBiconditional(f);
    const n2 = eliminateConditional(n1);
    const n3 = nnf(n2);
    const n4 = cnf(n3);
    return simplify(n4);
}

In [ ]:
function prettify<E>(M: Set<Set<E>>): string {
    if (M.size === 0) {
        return '{}';
    }
    let result = '{';
    for (const A of M) {
        if (A.size === 0) {
            result += '{}, ';
        } else {
            const elems: string[] = [];
            for (const lit of A) {
                if (typeof lit === 'string') {
                    elems.push(`'${lit}'`);
                } else {
                    elems.push(`('${lit[0]}', '${lit[1]}')`);
                } 
            }
            result += '{' + elems.join(', ') + '}, ';
        }
    }
    result = result.slice(0, -2); // remove trailing ", "
    result += '}';
    return result;
}

In [ ]:
function test(s: string): string {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    return prettify(normalize(f));
}

In [ ]:
test('(¬p → q) → (p → q) → q');

In [ ]:
test('(a → b) ↔ (¬a ∧ ¬b)');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p');

In [ ]:
test('⊤');

In [ ]:
test('⊥');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p');

In [ ]:
test("p → q");
test("(p ∧ q) → r");
test("p ↔ q");
test("(p → q) ∧ (q → r)");
test("¬(p ∨ q)");

In [ ]:
test('p ∧ p');